# Lab 4：Ascend C 典型算子开发

## 实验目标

- 理解 Host 侧通过 <<<>>> 启动 Ascend C Kernel 的完整路径。
- 使用 GM_ADDR 指针和基础类型参数在 Host 与 Kernel 间传递数据与切分信息。
- 补全数据搬入、计算、搬出的 3 处 Kernel 代码。
- 使用 FP32 输入完成 16384 个元素的精度校验。

## 实验环境

| 项目 | 配置 |
| --- | --- |
| NPU | 单卡 Ascend 910B3（Atlas A2） |
| CANN | 9.0.0 |
| Python | 3.11.4 |
| 关键工具 | Ascend C、BiSheng、AscendCL |

## 实验原理

本实验把 Host 代码和 Kernel 代码放在同一个 add_custom.asc 文件中。Host 侧负责准备输入、申请 Global Memory、启动 Kernel、同步 Stream 并校验结果；Kernel 侧在 AI Core 上完成 Global Memory 到 Local Memory 的搬运、矢量加法和结果写回。

Kernel 使用 GM_ADDR 表示 Global Memory 指针。除指针外，入口只接收 uint32_t 基础类型参数 totalLength 和 tileNum；Host 侧用 <<<>>> 显式指定启动 8 个 AIV 实例。

## 实验流程

### 1. 初始化实验环境

CANN 的命令行工具、头文件和动态库路径由 set_env.sh 配置。Notebook 的 !source 只在当前子 Shell 生效，因此下面的 Python 单元会把环境变量写回 Notebook 进程，并创建实验代码目录。

In [ ]:
!mkdir -p Sources/lab04

import os
import subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'",
    shell=True,
    text=True,
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("CANN 路径:", os.environ.get("ASCEND_HOME_PATH") or os.environ.get("ASCEND_TOOLKIT_HOME"))
print("\n环境初始化完成")

BiSheng 用于把同一个 ASC 源文件中的 Host 与 Kernel 代码编译为可执行程序。npu-smi info 用于查看设备状态。

In [ ]:
!which bisheng || echo "未找到 bisheng：set_env.sh 未生效"
!npu-smi info || echo "npu-smi 不可用"

### 2. 设计 Add 算子

本实验实现 FP32 逐元素加法：

z = x + y

| 项目 | 规格 |
| --- | --- |
| 输入 | x、y，均为 float |
| 输出 | z，float |
| 测试形状 | (8, 2048) |
| 元素总数 | 16384 |
| 启动核数 | 8 |
| 每核 Tile 数 | 8 |

16384 个元素先平均分给 8 个核，每核处理 2048 个元素；每核再分为 8 个 Tile，因此每个 Tile 有 256 个 float（1024 字节），满足单次搬运的 32 字节对齐要求。

### 3. 编写单文件程序

下面的 add_custom.asc 同时包含 Kernel 与 Host 程序。Kernel 的 Init 根据 totalLength、tileNum 和当前 Block 编号定位本核负责的连续数据段。每个 Tile 按 CopyIn、Compute、CopyOut 的顺序执行。

代码中有 3 处 TODO：

1. 将 x、y 的当前 Tile 从 Global Memory 搬入 Local Memory；
2. 在 Local Memory 上执行逐元素加法；
3. 将结果 Tile 写回 Global Memory。

Host 侧只传递设备指针与 uint32_t 参数，并通过 <<<>>> 发射 Kernel；不传递结构体参数。

In [ ]:
%%writefile Sources/lab04/add_custom.asc
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t kBlockDim = 8;
constexpr uint32_t kTileNum = 8;
constexpr uint32_t kElementCount = 8U * 2048U;
constexpr int32_t kBufferNum = 1;
constexpr int32_t kQueueDepth = 1;

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}

    __aicore__ inline void Init(
        GM_ADDR x,
        GM_ADDR y,
        GM_ADDR z,
        uint32_t totalLength,
        uint32_t tileNum)
    {
        blockLength_ = totalLength / AscendC::GetBlockNum();
        tileNum_ = tileNum;
        tileLength_ = blockLength_ / tileNum_;

        const uint32_t blockOffset =
            blockLength_ * AscendC::GetBlockIdx();
        xGm_.SetGlobalBuffer((__gm__ float *)x + blockOffset, blockLength_);
        yGm_.SetGlobalBuffer((__gm__ float *)y + blockOffset, blockLength_);
        zGm_.SetGlobalBuffer((__gm__ float *)z + blockOffset, blockLength_);

        pipe_.InitBuffer(inQueueX_, kBufferNum, tileLength_ * sizeof(float));
        pipe_.InitBuffer(inQueueY_, kBufferNum, tileLength_ * sizeof(float));
        pipe_.InitBuffer(outQueueZ_, kBufferNum, tileLength_ * sizeof(float));
    }

    __aicore__ inline void Process()
    {
        for (uint32_t progress = 0; progress < tileNum_; ++progress) {
            CopyIn(progress);
            Compute();
            CopyOut(progress);
        }
    }

private:
    __aicore__ inline void CopyIn(uint32_t progress)
    {
        AscendC::LocalTensor<float> xLocal =
            inQueueX_.AllocTensor<float>();
        AscendC::LocalTensor<float> yLocal =
            inQueueY_.AllocTensor<float>();

        // ---------------- TODO 1 / 3：搬入输入数据 ----------------
        // 将 xGm 和 yGm 的当前 Tile 搬入 Local Memory。
        // 当前 Tile 在本核数据段中的起始下标为 progress * tileLength_。
        // 在此处填写两行搬入代码。

        inQueueX_.EnQue(xLocal);
        inQueueY_.EnQue(yLocal);
    }

    __aicore__ inline void Compute()
    {
        AscendC::LocalTensor<float> xLocal =
            inQueueX_.DeQue<float>();
        AscendC::LocalTensor<float> yLocal =
            inQueueY_.DeQue<float>();
        AscendC::LocalTensor<float> zLocal =
            outQueueZ_.AllocTensor<float>();

        // ---------------- TODO 2 / 3：执行逐元素加法 ----------------
        // 使用 AscendC::Add 将 xLocal 与 yLocal 相加并写入 zLocal。
        // 在此处填写一行计算代码。

        outQueueZ_.EnQue(zLocal);
        inQueueX_.FreeTensor(xLocal);
        inQueueY_.FreeTensor(yLocal);
    }

    __aicore__ inline void CopyOut(uint32_t progress)
    {
        AscendC::LocalTensor<float> zLocal =
            outQueueZ_.DeQue<float>();

        // ---------------- TODO 3 / 3：搬出结果 ----------------
        // 将 zLocal 写回 zGm 中与当前 Tile 对应的位置。
        // 在此处填写一行搬出代码。

        outQueueZ_.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe_;
    AscendC::TQue<AscendC::TPosition::VECIN, kQueueDepth> inQueueX_;
    AscendC::TQue<AscendC::TPosition::VECIN, kQueueDepth> inQueueY_;
    AscendC::TQue<AscendC::TPosition::VECOUT, kQueueDepth> outQueueZ_;
    AscendC::GlobalTensor<float> xGm_;
    AscendC::GlobalTensor<float> yGm_;
    AscendC::GlobalTensor<float> zGm_;
    uint32_t blockLength_ = 0;
    uint32_t tileNum_ = 0;
    uint32_t tileLength_ = 0;
};

extern "C" __global__ __aicore__ void add_custom(
    GM_ADDR x,
    GM_ADDR y,
    GM_ADDR z,
    uint32_t totalLength,
    uint32_t tileNum)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, totalLength, tileNum);
    op.Process();
}

struct AclResources {
    ~AclResources()
    {
        Reset();
    }

    bool Initialize(size_t bytes)
    {
        if (aclInit(nullptr) != ACL_SUCCESS) {
            return false;
        }
        aclInitialized_ = true;
        if (aclrtSetDevice(0) != ACL_SUCCESS) {
            return false;
        }
        deviceSet_ = true;
        if (aclrtCreateStream(&stream) != ACL_SUCCESS) {
            return false;
        }
        if (aclrtMalloc(reinterpret_cast<void **>(&xDevice), bytes,
                ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS) {
            return false;
        }
        if (aclrtMalloc(reinterpret_cast<void **>(&yDevice), bytes,
                ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS) {
            return false;
        }
        return aclrtMalloc(reinterpret_cast<void **>(&zDevice), bytes,
                   ACL_MEM_MALLOC_HUGE_FIRST) == ACL_SUCCESS;
    }

    void Reset()
    {
        if (zDevice != nullptr) {
            aclrtFree(zDevice);
            zDevice = nullptr;
        }
        if (yDevice != nullptr) {
            aclrtFree(yDevice);
            yDevice = nullptr;
        }
        if (xDevice != nullptr) {
            aclrtFree(xDevice);
            xDevice = nullptr;
        }
        if (stream != nullptr) {
            aclrtDestroyStream(stream);
            stream = nullptr;
        }
        if (deviceSet_) {
            aclrtResetDevice(0);
            deviceSet_ = false;
        }
        if (aclInitialized_) {
            aclFinalize();
            aclInitialized_ = false;
        }
    }

    bool aclInitialized_ = false;
    bool deviceSet_ = false;
    aclrtStream stream = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;
};

int VerifyResult(
    const std::vector<float> &output,
    const std::vector<float> &golden)
{
    constexpr float kRtol = 1e-4F;
    constexpr float kAtol = 1e-4F;
    size_t mismatchCount = 0;
    float maxAbsDiff = 0.0F;

    std::printf("output[0..7] : ");
    for (size_t i = 0; i < 8; ++i) {
        std::printf("%.4f ", output[i]);
    }
    std::printf("\ngolden[0..7] : ");
    for (size_t i = 0; i < 8; ++i) {
        std::printf("%.4f ", golden[i]);
    }
    std::printf("\n");

    for (size_t i = 0; i < golden.size(); ++i) {
        const float diff = std::fabs(output[i] - golden[i]);
        maxAbsDiff = std::max(maxAbsDiff, diff);
        if (diff > kAtol + kRtol * std::fabs(golden[i])) {
            if (mismatchCount < 10) {
                std::printf(
                    "  mismatch at index %zu: output=%.6f golden=%.6f diff=%.6f\n",
                    i, output[i], golden[i], diff);
            }
            ++mismatchCount;
        }
    }

    std::printf("max abs diff : %.6f\n", maxAbsDiff);
    if (mismatchCount == 0) {
        std::printf("[PASS] 精度校验通过，共 %zu 个元素全部一致。\n",
                    golden.size());
        return 0;
    }
    std::printf("[FAIL] 精度校验未通过，%zu / %zu 个元素超出容差。\n",
                mismatchCount, golden.size());
    return 1;
}

int main()
{
    static_assert(kElementCount % kBlockDim == 0,
                  "element count must divide evenly across blocks");
    static_assert((kElementCount / kBlockDim) % kTileNum == 0,
                  "each block must divide evenly into tiles");

    std::vector<float> x(kElementCount);
    std::vector<float> y(kElementCount);
    std::vector<float> output(kElementCount, 0.0F);
    std::vector<float> golden(kElementCount);

    for (uint32_t i = 0; i < kElementCount; ++i) {
        x[i] = static_cast<float>(i % 97U) * 0.5F;
        y[i] = static_cast<float>(i % 53U) * 0.25F;
        golden[i] = x[i] + y[i];
    }

    const size_t bytes = static_cast<size_t>(kElementCount) * sizeof(float);
    AclResources resources;
    if (!resources.Initialize(bytes)) {
        std::printf("ACL 初始化或显存分配失败。\n");
        return 1;
    }

    if (aclrtMemcpy(resources.xDevice, bytes, x.data(), bytes,
            ACL_MEMCPY_HOST_TO_DEVICE) != ACL_SUCCESS ||
        aclrtMemcpy(resources.yDevice, bytes, y.data(), bytes,
            ACL_MEMCPY_HOST_TO_DEVICE) != ACL_SUCCESS) {
        std::printf("输入数据拷贝失败。\n");
        return 1;
    }

    add_custom<<<kBlockDim, nullptr, resources.stream>>>(
        resources.xDevice,
        resources.yDevice,
        resources.zDevice,
        kElementCount,
        kTileNum);

    if (aclrtSynchronizeStream(resources.stream) != ACL_SUCCESS) {
        std::printf("Kernel 执行失败。\n");
        return 1;
    }

    if (aclrtMemcpy(output.data(), bytes, resources.zDevice, bytes,
            ACL_MEMCPY_DEVICE_TO_HOST) != ACL_SUCCESS) {
        std::printf("结果数据拷贝失败。\n");
        return 1;
    }

    return VerifyResult(output, golden);
}


### 4. 编译程序

ASC 后缀会使 BiSheng 按 Ascend C 编译。--npu-arch=dav-2201 是本课程 Ascend 910B3（Atlas A2）的基线架构；若实验环境不同，应先记录实际硬件与 CANN 版本，再按环境要求调整该参数。

In [ ]:
!bisheng Sources/lab04/add_custom.asc -o Sources/lab04/execute_add_op --npu-arch=dav-2201
!ls -l Sources/lab04/execute_add_op

### 5. 运行并校验精度

程序生成互不相同的 FP32 输入，避免固定输入掩盖 Tile 偏移错误。Host 在 Kernel 发射后调用 aclrtSynchronizeStream，随后把输出拷回并逐元素与 CPU 标准结果比较。比较容差为 rtol = atol = 1e-4。

In [ ]:
!Sources/lab04/execute_add_op

预期输出：

<pre>
output[0..7] : 0.0000 0.7500 1.5000 2.2500 3.0000 3.7500 4.5000 5.2500
golden[0..7] : 0.0000 0.7500 1.5000 2.2500 3.0000 3.7500 4.5000 5.2500
max abs diff : 0.000000
[PASS] 精度校验通过，共 16384 个元素全部一致。
</pre>

若精度校验失败，先检查 3 个 TODO 中的 Tile 偏移量是否都使用 progress * tileLength_，以及 CopyOut 是否写回 zGm 的对应位置。

## 实验总结

本实验完成了一个 FP32 Add Kernel。Host 侧使用 AscendCL 管理设备内存与 Stream，并以 <<<>>> 启动 8 个核；Kernel 侧使用 GM_ADDR 指针和 uint32_t 切分参数，将数据按核和 Tile 分配后完成搬运、加法和写回。

## 实验扩展

1. 为什么 Kernel 入口中的 totalLength 和 tileNum 使用 uint32_t，而不是传递自定义结构体？
2. 16384 个元素如何逐级得到每 Tile 的 256 个 float？
3. CopyIn 与 CopyOut 为什么都需要使用 progress * tileLength_ 作为核内偏移？
4. 为什么 Kernel 发射后必须在读回结果前同步 Stream？
5. 如果把测试长度改为其他值，Host 侧需要满足哪些分块和对齐条件？

运行下一单元查看参考答案。

In [ ]:
!cat answer/thought_questions.txt

## 参考答案

参考答案给出了完整的单文件程序，包含 3 处 TODO 的实现。

In [ ]:
!cat answer/04.01_answer/add_custom.asc

如需使用参考实现重新编译和验证，可执行下一单元覆盖实验源码，然后重新运行编译与精度校验单元。

In [ ]:
!cp answer/04.01_answer/add_custom.asc Sources/lab04/add_custom.asc